# `passage-eval` — Walkthrough 03: leaderboard visualisation

Aggregates everything `01_astar_walkthrough.ipynb` and
`02_score_released_baselines.ipynb` emitted under `results/runs/` and
produces the four canonical plots:

1. **Bar chart** of `cost_regret_median` per method (lower is better).
2. **Bar chart** of `cost_regret_p90` per method.
3. **Pareto scatter** of accuracy vs end-to-end latency (Thor 256$\times$256).
4. **Trustworthiness radar** (raw axis values) for the methods that have them.

## 1. Imports + load the leaderboard

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PE_SRC = Path('~/projects/passage/passage/src/passage-eval/src')
if str(PE_SRC) not in sys.path: sys.path.insert(0, str(PE_SRC))
import passage_eval as pe

RESULTS_ROOT = Path('~/projects/passage/passage/src/passage-eval/results')
lb = pe.load_leaderboard(RESULTS_ROOT)
print('total rows:', len(lb))
lb.df.groupby(['method_name','family']).size().rename('n_seeds').reset_index()


total rows: 308


,method_name,family,n_seeds
0,astar_classical,classical,28
1,graph-edge_mlp_classical_astar,graph,84
2,greedy_classical,classical,28
3,reg-repvit_m1_bellman_vbridge,dense_regression,84
4,seg-convnextv2_atto_baseline,dense_segmentation,84


## 2. Best-per-method headline table

In [2]:
best = lb.best_per_method(metric='cost_regret_median', higher_is_better=False)
cols = ['method_name','family','architecture','decoder_variant',
        'resolution','cost_model','seed','n_samples',
        'cost_regret_median','cost_regret_p90','path_assd','path_hausdorff_p95',
        'success_rate','latency_e2e_ms_p50']
best[cols].sort_values('cost_regret_median')


,method_name,family,architecture,decoder_variant,resolution,cost_model,seed,n_samples,cost_regret_median,cost_regret_p90,path_assd,path_hausdorff_p95,success_rate,latency_e2e_ms_p50
0,astar_classical,classical,analytic,analytic_a_star,32,elevation,1234,8,0.000000,0.000000,0.000000,0.000000,1.00000,0.500000
3,reg-repvit_m1_bellman_vbridge,dense_regression,repvit_m1,V_strict,256,elevation,3407,1024,0.001018,0.020623,0.022701,0.057647,0.99707,10.790000
4,seg-convnextv2_atto_baseline,dense_segmentation,convnextv2_atto,BCE_threshold,256,elevation,1234,1024,0.036936,0.142076,0.110557,0.196772,0.99707,11.270000
2,greedy_classical,classical,analytic,manhattan_descent,32,elevation,1234,8,0.087332,0.123439,0.028163,0.054626,1.00000,0.050000
1,graph-edge_mlp_classical_astar,graph,edge_mlp_5_16_16_1,analytic_a_star_on_learned_costs,256,elevation,1234,64,0.113648,0.244158,0.082523,0.191006,1.00000,51.600044


## 3. Bar charts of headline metrics

In [3]:
# Restrict to 256x256 + elevation rows so the synthetic 32x32 classical entries don't dominate.
df256 = lb.df[(lb.df.resolution == 256) & (lb.df.cost_model == 'elevation')]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pe.plot_primary_bars(df256, metric='cost_regret_median', ax=axes[0])
pe.plot_primary_bars(df256, metric='cost_regret_p90',    ax=axes[1])
fig.suptitle('PASSAGE 256x256 / elevation — leaderboard', fontsize=12)
fig.tight_layout()
plt.show()


## 4. Pareto scatter: accuracy vs end-to-end latency

In [4]:
fig, ax = plt.subplots(figsize=(7.5, 5.0))
pe.plot_pareto_accuracy_latency(df256,
                                 accuracy_metric='cost_regret_median',
                                 latency_column='latency_e2e_ms_p50',
                                 ax=ax)
plt.show()


## 5. Trustworthiness radar

For each method that has 3 seeds (so we can compute Stab) we read the
per-sample arrays, derive three of the four axes (Stab from CR-p90 CV
across seeds, ODDe from the obstacle-coverage stratifier on the
primary seed, Eff from the released latency), and plot them on one
shared radar.

In [5]:
def trustworthiness_for_method(df_method):
    seeds = df_method['seed'].tolist()
    cr_p90 = df_method['cost_regret_p90'].tolist()
    stab = pe.stability_cv(cr_p90)
    eff = pe.oracle_relative_efficiency(
        df_method['latency_e2e_ms_p50'].iloc[0],
        50.0,  # Thor CPU oracle reference (App. I)
    )
    # Find a per-sample NPZ for ODDe (any seed will do).
    npz_pointer = df_method.iloc[0].get('per_sample_path')
    odde_value = float('nan')
    if isinstance(npz_pointer, str) and npz_pointer:
        npz_path = RESULTS_ROOT / npz_pointer
        if npz_path.exists():
            arrays = pe.load_per_sample_npz(npz_path)
            odde = pe.odd_boundary_delta(arrays.regret, arrays.obs_density, top_quantile=0.10)
            odde_value = odde.value
    return {'method_name': df_method.iloc[0]['method_name'],
            'Stab': stab.value,
            'Geo': float('nan'),       # Geo axis needs a matched-marginal random split (not on disk yet).
            'ODDe': odde_value,
            'Eff': eff.value}

rows = []
for method, sub in df256.groupby('method_name'):
    if sub['seed'].nunique() < 2: continue
    rows.append(trustworthiness_for_method(sub))
print(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw={'projection': 'polar'})
pe.plot_trustworthiness_radar(rows, ax=ax,
                               title='Trustworthiness radar — Stab / Geo / ODDe / Eff')
plt.tight_layout()
plt.show()


                      method_name      Stab  Geo  ODDe       Eff
0  graph-edge_mlp_classical_astar  0.029930  NaN   NaN  0.968991
1   reg-repvit_m1_bellman_vbridge  0.086098  NaN   NaN  4.633920
2    seg-convnextv2_atto_baseline  0.008861  NaN   NaN  4.436557


## 6. Pairwise verdict heatmap

In [6]:
# Pairwise reg-vbridge vs seg-bce, multiseed paired bootstrap on the censored regret arrays.
def collect_regret(method_label):
    rows = lb.df[lb.df['method_name'] == method_label]
    # Dedupe to one row per seed (the metric arrays are seed-invariant
    # across accelerators; the per_sample.npz file is the same).
    rows = rows.drop_duplicates(subset='seed').sort_values('seed')
    arrays = []
    for _, r in rows.iterrows():
        if not isinstance(r.get('per_sample_path'), str):
            return None
        npz_path = RESULTS_ROOT / r['per_sample_path']
        if not npz_path.exists():
            return None
        with np.load(npz_path) as store:
            arrays.append(np.asarray(store['regret']))
    return arrays if len(arrays) == 3 else None

reg = collect_regret('reg-repvit_m1_bellman_vbridge')
seg = collect_regret('seg-convnextv2_atto_baseline')

decisions = {}
if reg is not None and seg is not None:
    delta_med = pe.multiseed_paired_bootstrap(reg, seg, metric='cost_regret_median',
                                                statistic=np.nanmedian, n_bootstrap_per_seed=1000, alpha=0.05)
    delta_p90 = pe.multiseed_paired_bootstrap(reg, seg, metric='cost_regret_p90',
                                                statistic=lambda x: float(np.nanquantile(x, 0.90)),
                                                n_bootstrap_per_seed=1000, alpha=0.05)
    deltas = [delta_med, delta_p90]
    holm = pe.holm_bonferroni({d.metric: d.two_sided_p_value() for d in deltas}, alpha=0.05)
    verdict, votes = pe.decide_pairwise(deltas, holm=holm)
    decisions[('reg-repvit_m1_bellman_vbridge', 'seg-convnextv2_atto_baseline')] = str(verdict).split('.')[-1]
    print('verdict:', verdict)
    print('votes  :', votes)
else:
    print('paired bootstrap skipped — per-sample arrays not found for at least one method')

fig, ax = plt.subplots(figsize=(6, 5))
pe.plot_pairwise_heatmap(decisions, ax=ax)
plt.tight_layout()
plt.show()


verdict: PairwiseDecision.TRADE_OFF
votes  : {'cost_regret_median': 'win_a', 'cost_regret_p90': 'win_b'}


## 7. Summary

The leaderboard is now fully observable from a single CSV
(`results/leaderboard.csv`) plus per-run bundles (`results/runs/*/`).
The four canonical plots are produced by `passage_eval.plot_*`
helpers — no extra dependencies beyond matplotlib. A new submission
appends one row + one bundle; the leaderboard view picks it up
immediately.

The same notebooks compose: `01_*` shows how to add a *new* planner,
`02_*` shows how to register pre-computed predictions, `03_*` is the
read-only consumer.